# ML-04 ? Search Intelligence Data Contract

**Lane 2: Refresh / Content Opportunity Scoring**

This notebook uses the full pseudonymized FlyRank warehouse. It reads only the March and April 2026 partitions; the June 2026 sample/final month remains sealed. No raw warehouse data is saved or committed.

## 1. Unit of analysis + time window

Here are the five contract answers in plain words:

1. **What one row means:** the source table has one row per pseudonymized client ? content page ? day. My modeling frame aggregates that to one row per client ? content page at the **2026-04-01 decision moment**.
2. **Tables used:** `fact_content_daily_performance` only, using its `month=2026-03` and `month=2026-04` partitions. I do not use the June `_sample` table.
3. **Time window:** features are measured from **2026-03-01 through 2026-03-31**. The outcome is measured later, from **2026-04-01 through 2026-04-30**.
4. **What I rank:** I estimate whether a page's April GSC impressions fall more than 20% below its March impressions, then rank pages by that decline-risk score for human refresh review. This is a measured **proxy**, not proof that a refresh will help.
5. **What I deliberately exclude:** April impressions, the April/March change, IDs, product decision flags, and anything from June are excluded from features. They are future information, context-only identifiers, circular product outputs, or sealed-test information.

## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| Feature | `log_march_impressions`, `march_ctr`, `march_avg_position`, `march_active_days`, `march_impression_momentum` | All are calculated only from measured March rows and are available by 2026-04-01. |
| Label / proxy | `is_next_month_decline`; its source is `april_impressions < 0.8 ? march_impressions` | This is the future measured outcome I try to rank, so neither it nor its source can be a feature. |
| Context | `client_hash_id`, `content_hash_id` | Needed for grain, joins, and client-grouped validation; never shown to the model. |
| Excluded | April raw metrics and April/March change; June `_sample`; product flags; GA4 zero-filled values; pages with fewer than 100 March impressions | These respectively leak the answer, peek at the sealed month, copy a product decision, confuse unavailable tracking with zero activity, or make a 20% change too noisy. |

Missing-value rule: the lane uses GSC features and keeps only rows where `gsc_data_available IS TRUE`. The availability check below also measures GA4 coverage with `ga4_data_available IS TRUE`; GA4 zero-filled values are not used as features.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass, duckdb
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except (ImportError, ModuleNotFoundError, KeyError, RuntimeError):
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (input hidden): ")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required; never paste it into a notebook cell.")

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])
REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
print("Connected securely; only March and April partitions are in scope.")


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Connected securely; only March and April partitions are in scope.


## 3. Three verification queries

These are the **exactly three** contract-verification queries: grain, count/date span, and measurement availability. All outputs are aggregate and public-safe.

In [2]:
# Verification query 1 of 3 ? source grain. Zero duplicate keys proves page-day grain.
grain_check = con.sql(f"""
    SELECT COUNT(*) AS duplicate_page_day_keys
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {MARCH}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    ) duplicates
""").df()
grain_check

,duplicate_page_day_keys
0,0


In [3]:
# Verification query 2 of 3 ? lane slice size and observed date span.
slice_check = con.sql(f"""
    SELECT COUNT(*) AS page_day_rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id)) AS distinct_pages,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {MARCH}
""").df()
slice_check

,page_day_rows,distinct_pages,first_date,last_date
0,9841378,331437,2026-03-01,2026-03-31


In [4]:
# Verification query 3 of 3 ? availability. IS TRUE handles both FALSE and NULL safely.
availability_check = con.sql(f"""
    SELECT COUNT(*) AS all_page_day_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows_surviving,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows_surviving,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                              AND ga4_data_available IS TRUE) AS both_rows_surviving
    FROM {MARCH}
""").df()
availability_check

,all_page_day_rows,gsc_rows_surviving,ga4_rows_surviving,both_rows_surviving
0,9841378,3611061,413966,364347


### Five-feature frame

Exactly five model features are kept:

1. `log_march_impressions` ? knowable at the decision moment because all March impression rows have landed by 2026-04-01.
2. `march_ctr` ? knowable at the decision moment because it uses only March clicks divided by March impressions.
3. `march_avg_position` ? knowable at the decision moment because it averages only observed March GSC positions on impression-active days.
4. `march_active_days` ? knowable at the decision moment because it counts March days with at least one observed impression.
5. `march_impression_momentum` ? knowable at the decision moment because it compares March 17?31 impressions with March 1?16 impressions; both halves end before scoring.

In [5]:
# Feature-building query (not a contract-verification query): aggregate remotely, return one row per page.
FEATURE_SQL = f"""
WITH march AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS march_impressions,
           SUM(gsc_clicks) AS march_clicks,
           AVG(gsc_avg_position) FILTER (WHERE gsc_impressions > 0) AS march_avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS march_active_days,
           SUM(CASE WHEN report_date < DATE '2026-03-17' THEN gsc_impressions ELSE 0 END) AS early_impressions,
           SUM(CASE WHEN report_date >= DATE '2026-03-17' THEN gsc_impressions ELSE 0 END) AS late_impressions
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
april AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS april_impressions
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id,
       LN(1 + m.march_impressions) AS log_march_impressions,
       m.march_clicks::DOUBLE / NULLIF(m.march_impressions, 0) AS march_ctr,
       m.march_avg_position,
       m.march_active_days,
       (m.late_impressions + 1.0) / (m.early_impressions + 1.0) AS march_impression_momentum,
       m.march_impressions, a.april_impressions,
       (a.april_impressions < 0.8 * m.march_impressions)::INTEGER AS is_next_month_decline
FROM march m
INNER JOIN april a USING (client_hash_id, content_hash_id)
"""
feature_frame = con.sql(FEATURE_SQL).df()
FEATURES = ["log_march_impressions", "march_ctr", "march_avg_position",
            "march_active_days", "march_impression_momentum"]
assert len(FEATURES) == 5
assert not feature_frame.duplicated(["client_hash_id", "content_hash_id"]).any()
print(f"Modeling frame: {len(feature_frame):,} unique pages; decline proxy rate: {feature_frame['is_next_month_decline'].mean():.1%}")
feature_frame[FEATURES + ["is_next_month_decline"]].head()

Modeling frame: 100,893 unique pages; decline proxy rate: 51.5%


,log_march_impressions,march_ctr,march_avg_position,march_active_days,march_impression_momentum,is_next_month_decline
0,5.963579,0.000000,5.629009,31,0.805556,1
1,5.214936,0.000000,4.154013,31,0.813725,1
2,6.982863,0.000929,1.432291,31,0.670279,1
3,6.204558,0.002024,6.844568,31,0.471810,1
4,8.964184,0.002047,2.267155,31,1.074556,0


### The deliberate leakage trap

I first fit the honest five-feature tree with a client-grouped holdout. Then I intentionally add `future_impression_ratio = April impressions / March impressions`. Because the label is literally `future_impression_ratio < 0.8`, this column is the answer in disguise. Its near-perfect score is not useful performance; it is leakage. The cell deletes it and retains the honest score.

In [6]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

y = feature_frame["is_next_month_decline"]
groups = feature_frame["client_hash_id"]
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(feature_frame, y, groups))

def quick_tree(columns):
    model = make_pipeline(SimpleImputer(strategy="median"),
                          DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42))
    model.fit(feature_frame.iloc[train_idx][columns], y.iloc[train_idx])
    return accuracy_score(y.iloc[test_idx], model.predict(feature_frame.iloc[test_idx][columns]))

honest_accuracy = quick_tree(FEATURES)
feature_frame["future_impression_ratio"] = feature_frame["april_impressions"] / feature_frame["march_impressions"]
leaky_accuracy = quick_tree(FEATURES + ["future_impression_ratio"])
print(f"Honest 5-feature grouped-holdout accuracy: {honest_accuracy:.3f}")
print(f"LEAKY accuracy after adding future_impression_ratio: {leaky_accuracy:.3f}")
print("The jump is leakage: future_impression_ratio directly defines the label.")
feature_frame.drop(columns="future_impression_ratio", inplace=True)
assert "future_impression_ratio" not in feature_frame.columns
assert len(FEATURES) == 5
print(f"Leak removed. Final reported accuracy remains: {honest_accuracy:.3f}")

Honest 5-feature grouped-holdout accuracy: 0.659
LEAKY accuracy after adding future_impression_ratio: 1.000
The jump is leakage: future_impression_ratio directly defines the label.
Leak removed. Final reported accuracy remains: 0.659


## 4. Named limitation

**Limitation ? survivor/coverage bias from an unbalanced panel:** this slice requires a page to have at least 100 measured March impressions and an observed April row. It therefore excludes low-volume pages, pages that disappear before April, and clients without usable history in both windows. The result can support prioritization among eligible observed pages, but it does not represent every content page and cannot show that refreshing a flagged page would cause recovery.

## 5. Self-check

- [x] Five plain-words contract answers are present.
- [x] Exactly three verification queries are executed with visible outputs.
- [x] Availability is filtered with `IS TRUE`.
- [x] The modeling frame has exactly five decision-time features, each with an availability line.
- [x] One label-derived feature is deliberately added, scored, deleted, and excluded from the final list.
- [x] One named limitation is stated.
- [x] No client names, URLs, raw search queries, tokens, or raw warehouse data are exposed.
- [x] June 2026 remains a sealed test month.
- [x] Claims are framed as measured, directional, and decision-support rather than causal.